# Ultrasound Agent -- unified inference across organs

The Ultrasound Agent as the system architecture defines it: **it reads an image and reports what it
sees, with how confident it is.** It routes by organ, returns findings plus calibrated confidence
plus an explainability reference, and writes the result into a patient encounter bundle that the
Clinical Reasoning Agent later reads.

**There is no training here.** Training lives in the four module notebooks, which are frozen. This
notebook loads their saved checkpoints and runs inference, so it completes in minutes rather than
hours.

| Organ | Module | Status |
|---|---|---|
| Heart | EfficientNet-B0 U-Net, CAMUS | supported |
| Gallbladder | EfficientNet-B0, 5 classes | supported |
| Lung | EfficientNet-B0, 4 findings, multi-label | supported |
| Vascular / FAST | — | `not_supported`, same schema |

## What this agent does not do

No urgency. No diagnosis. No treatment. Those require the vitals, laboratory results and history
that this agent never sees, and they belong to the Clinical Reasoning Agent, which fuses this
output with the Triage Agent's severity assessment.

A radiologist writes *"multiple B-lines, small consolidation, no pleural effusion"*. The leap to
*"pulmonary oedema"* is a different job, done by a different agent, with more information.

## Flow

```
   image + organ                encounters/ENC-001.json
        |                                ^
        v                                |
  route by organ  -->  findings  -->  write section
        |               confidence
        |               grad-cam
        v
  unsupported -> status: "not_supported", same schema
```

Every organ returns the identical shape, so the reasoning agent never special-cases: an
unsupported organ is an empty findings list with a status, not a missing key.


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q segmentation_models_pytorch opencv-python-headless


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.3 MB/s eta 0:00:00


In [2]:
import io
import json
import sys
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image

DRIVE_ROOT = Path('/content/drive/MyDrive/POCUS-Project')
ENCOUNTER_ROOT = DRIVE_ROOT / 'encounters_store'

# The shared contract lives in the repo so all four notebooks emit the same shape. Upload src/
# to Drive alongside the data; without it this notebook cannot guarantee schema consistency,
# which is the entire point of having one.
sys.path.insert(0, str(DRIVE_ROOT))
try:
    from src.agents import schema as S
    print('Loaded shared schema, version', S.SCHEMA_VERSION)
except ModuleNotFoundError:
    raise SystemExit(
        'src/agents/schema.py not found on Drive. Upload the repo src/ folder to '
        f'{DRIVE_ROOT}/src so every organ emits the same contract.')

IMG_SIZE_CLS = 224      # gallbladder / lung classifiers
IMG_SIZE_SEG = 256      # cardiac segmentation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Inference still works, just slower.')


Loaded shared schema, version 1.0
Device: cuda


## 2. Artifacts

Each module contributes two things: **weights**, and the **calibration it was fitted with**.

The second matters as much as the first. Every module's reported confidence is a calibrated
probability -- cardiac by isotonic regression, gallbladder by temperature scaling, lung by
per-finding logistic regression -- and those calibrators live only in the training notebooks'
memory. Loading weights without them would emit raw sigmoid or raw agreement scores, which are
**not** comparable across organs and would be silently wrong rather than obviously wrong.

So: if a calibration file is missing, that organ still runs, but its output carries
`confidence_calibrated: false`. The reasoning agent can then discount it instead of treating an
uncalibrated number as if it meant the same thing as a calibrated one.

Section 3 prints exactly which artifacts are present, and Section 9 lists the export snippets to
add to the module notebooks for anything missing.


In [3]:
ARTIFACTS = {
    'heart': {
        'weights': DRIVE_ROOT / 'Cardiac' / 'cardiac_effnet_unet_best.pth',
        'calib':   DRIVE_ROOT / 'Cardiac' / 'cardiac_calibration.json',
    },
    'gallbladder': {
        'weights': DRIVE_ROOT / 'Abdominal_Gallbladder' / 'gallbladder_effnetb0_best.pth',
        'calib':   DRIVE_ROOT / 'Abdominal_Gallbladder' / 'gallbladder_calibration.json',
    },
    'lung': {
        'weights': DRIVE_ROOT / 'Pulmonary' / 'lung_finding_classifier_efficientnet_b0_split0_best.pth',
        'calib':   DRIVE_ROOT / 'Pulmonary' / 'lung_calibration.json',
    },
}

print(f'{"organ":<14}{"weights":>10}{"calibration":>14}')
print('-' * 38)
AVAILABLE = {}
for organ, paths in ARTIFACTS.items():
    w, c = paths['weights'].exists(), paths['calib'].exists()
    AVAILABLE[organ] = {'weights': w, 'calib': c}
    print(f'{organ:<14}{"yes" if w else "MISSING":>10}{"yes" if c else "missing":>14}')

missing_w = [o for o, a in AVAILABLE.items() if not a['weights']]
missing_c = [o for o, a in AVAILABLE.items() if a['weights'] and not a['calib']]
if missing_w:
    print(f'\n{missing_w} have no weights -- they will report status="not_supported".')
if missing_c:
    print(f'{missing_c} have weights but no calibration -- confidence will be reported RAW')
    print('and flagged confidence_calibrated=false. See Section 9 to export it.')


organ            weights   calibration
--------------------------------------
heart                yes           yes
gallbladder          yes           yes
lung                 yes           yes


## 3. Load the models

Architectures are redeclared here to match each module's saved checkpoint. Nothing is trained.


In [4]:
import segmentation_models_pytorch as smp

CARDIAC_STRUCTURES = {0: 'background', 1: 'LV', 2: 'myocardium', 3: 'LA'}
GB_CLASS_NAMES = {0: 'Cholelithiasis', 1: 'Acute cholecystitis (any severity)',
                  2: 'Polyps / adenomyomatosis', 3: 'Carcinoma', 4: 'Wall thickening'}
GB_GROUP = {0: 'Cholelithiasis', 1: 'Acute inflammation', 2: 'Wall thickening / mass',
            3: 'Wall thickening / mass', 4: 'Wall thickening / mass'}
# The gallbladder model predicts 8 fine-grained classes and marginalises to these 5 (see its
# notebook): merging predictions rather than labels scored 63.1% against 57.2%.
GB_TRAIN_TO_EVAL = {0: 0, 1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 4}
GB_TRAIN_NAMES = {0: 'Gallstones', 1: 'Cholecystitis', 2: 'Membranous / gangrenous',
                  3: 'Perforation', 4: 'Polyps', 5: 'Adenomyomatosis',
                  6: 'Carcinoma', 7: 'Wall thickening'}
LUNG_FINDINGS = ['b_lines', 'consolidation', 'pleural_effusion', 'pleural_thickening']


class LungFindingClassifier(nn.Module):
    def __init__(self, feats, n_ftrs, n_labels, dropout=0.4, hidden=512):
        super().__init__()
        self.features = feats
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(n_ftrs, hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(hidden, n_labels))

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


MODELS = {}


def _load(organ, build_fn):
    if not AVAILABLE[organ]['weights']:
        return
    m = build_fn()
    m.load_state_dict(torch.load(ARTIFACTS[organ]['weights'], map_location=device))
    MODELS[organ] = m.to(device).eval()
    print(f'{organ:<14} loaded ({sum(p.numel() for p in m.parameters()):,} params)')


def _cardiac():
    return smp.Unet(encoder_name='efficientnet-b0', encoder_weights=None,
                    in_channels=1, classes=len(CARDIAC_STRUCTURES))


def _gallbladder():
    m = torchvision.models.efficientnet_b0(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(GB_TRAIN_NAMES))
    return m


def _lung():
    e = torchvision.models.efficientnet_b0(weights=None)
    return LungFindingClassifier(nn.Sequential(e.features, e.avgpool),
                                 e.classifier[1].in_features, len(LUNG_FINDINGS))


_load('heart', _cardiac)
_load('gallbladder', _gallbladder)
_load('lung', _lung)
print(f'\n{len(MODELS)} organ model(s) ready:', sorted(MODELS))


heart          loaded (6,251,328 params)
gallbladder    loaded (4,017,796 params)
lung           loaded (4,665,472 params)

3 organ model(s) ready: ['gallbladder', 'heart', 'lung']


## 4. Calibration

Loaded from each module's exported file. Where a file is absent the organ falls back to raw
scores and says so in its payload rather than pretending.


In [5]:
CALIB = {}
for organ, paths in ARTIFACTS.items():
    if AVAILABLE[organ]['calib']:
        CALIB[organ] = json.loads(paths['calib'].read_text())

print('calibration loaded for:', sorted(CALIB) or 'nothing')


def apply_temperature(logits, T):
    return torch.softmax(torch.as_tensor(logits) / T, dim=-1).numpy()


def apply_platt(p, coef, intercept):
    """Logistic map fitted per lung finding: sigmoid(coef * p + intercept)."""
    return float(1.0 / (1.0 + np.exp(-(coef * p + intercept))))


def apply_isotonic(x, xs, ys):
    """Piecewise-linear interpolation of the isotonic fit exported by the cardiac notebook."""
    return float(np.interp(x, xs, ys))


calibration loaded for: ['gallbladder', 'heart', 'lung']


## 5. Cardiac

Segments LV, myocardium and LA, then derives ejection fraction from the predicted masks using the
area-to-volume correction established in that module (raw area EF under-states the fractional
change because volume scales as area^(3/2); correcting it cut MAE from 12.7pp to 7.1pp).

Confidence is the fraction of eight fixed test-time augmentations that agree on the reported band
-- deliberately fixed rather than random, so the same clip always yields the same number.


In [6]:
EF_FINDING = {'normal': 'Normal ventricular function',
              'reduced': 'Mild-to-moderate LV dysfunction',
              'severe': 'Severe LV dysfunction'}


def _ef_band(ef):
    if ef >= 55: return 'normal'
    if ef >= 30: return 'reduced'
    return 'severe'


def _prep_seg(img):
    a = cv2.resize(np.asarray(img, dtype=np.float32), (IMG_SIZE_SEG, IMG_SIZE_SEG),
                   interpolation=cv2.INTER_LINEAR)
    if a.max() > 1.5:
        a = a / 255.0
    return a.astype(np.float32)


def _tta(img):
    h, w = img.shape
    out = [img.copy()]
    for ang in (3.0, -3.0):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
        out.append(cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=0))
    for b in (1.05, 0.95):
        out.append(np.clip(img * b, 0, 1))
    for c in (1.05, 0.95):
        m = img.mean(); out.append(np.clip((img - m) * c + m, 0, 1))
    out.append(np.fliplr(img).copy())
    return [a.astype(np.float32) for a in out]


def _lv_area(a):
    x = torch.from_numpy(a).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        return float((MODELS['heart'](x).argmax(1).squeeze(0).cpu().numpy() == 1).sum())


def predict_heart(ed_img, es_img):
    if 'heart' not in MODELS:
        return S.make_report('heart', [], status='not_supported')

    ed, es = _prep_seg(ed_img), _prep_seg(es_img)
    efs = []
    for a, b in zip(_tta(ed), _tta(es)):
        a_ed, a_es = _lv_area(a), _lv_area(b)
        if a_ed > 0:
            r = min(max(1.0 - (a_ed - a_es) / a_ed, 1e-6), 1.0)   # area -> volume correction
            efs.append((1.0 - r ** 1.5) * 100.0)

    if not efs:
        return S.make_report('heart', [], status='failed',
                             quality={'lv_detected': False})

    bands = [_ef_band(e) for e in efs]
    band = max(set(bands), key=bands.count)
    raw = bands.count(band) / len(bands)

    cal = CALIB.get('heart')
    conf = apply_isotonic(raw, cal['isotonic_x'], cal['isotonic_y']) if cal else raw

    return S.make_report(
        'heart',
        [S.make_finding(EF_FINDING[band], conf)],
        measurements={'ejection_fraction': round(float(np.median(efs)), 1),
                      'ef_spread_pp': round(float(np.std(efs)), 1)},
        quality={'lv_detected': True},
        reliability={'confidence_calibrated': bool(cal),
                     'confidence_ceiling': (cal or {}).get('ceiling', 1.0),
                     'scope': 'CAMUS-like 4CH stills; EF is an area proxy, not volumetric'},
        explanation={'type': 'segmentation_mask'},
        model='effnetb0_unet')


## 6. Gallbladder

Predicts eight fine-grained classes and marginalises to five reported ones by **summing** group
probabilities rather than mapping the argmax -- a case that splits its mass across cholecystitis,
gangrenous and perforation should report the total, not the largest single component.

`has_normal_class` is false and that is not a detail: the training data contained no healthy
gallbladders, so every prediction is a choice among pathologies and none of them means "no
disease".


In [7]:
IMAGENET_MEAN, IMAGENET_STD = 0.449, 0.226


def _prep_cls(img):
    a = cv2.resize(np.asarray(img), (IMG_SIZE_CLS, IMG_SIZE_CLS), interpolation=cv2.INTER_AREA)
    if a.ndim == 3:
        a = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    a = a.astype(np.float32)
    if a.max() > 1.5:
        a = a / 255.0
    a = (a - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(a).float().unsqueeze(0).repeat(3, 1, 1)


def predict_gallbladder(img):
    if 'gallbladder' not in MODELS:
        return S.make_report('gallbladder', [], status='not_supported')

    x = _prep_cls(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = MODELS['gallbladder'](x).cpu().numpy()[0]

    cal = CALIB.get('gallbladder')
    p8 = apply_temperature(logits, cal['temperature']) if cal else \
        torch.softmax(torch.from_numpy(logits), dim=-1).numpy()

    p5 = np.zeros(len(GB_CLASS_NAMES), dtype=np.float32)
    for t, e in GB_TRAIN_TO_EVAL.items():          # marginalise, do not argmax-then-map
        p5[e] += p8[t]
    c = int(p5.argmax())

    return S.make_report(
        'gallbladder',
        [S.make_finding(GB_CLASS_NAMES[c], float(p5[c]), group=GB_GROUP[c])],
        quality={'low_confidence': bool(p5[c] < 0.4)},
        reliability={'confidence_calibrated': bool(cal),
                     'has_normal_class': False,
                     'fine_grained': GB_TRAIN_NAMES[int(p8.argmax())],
                     'scope': 'teaching-atlas stills; no healthy class exists in the training data'},
        explanation={'type': 'grad_cam'},
        model='effnetb0')


## 7. Lung

The multi-label module, and the reason the shared schema carries a **list** of findings: B-lines
and an effusion co-occur, so a single-label contract could not represent a real clip.

Frame probabilities reduce to one score per clip by **median**, matching how the module was
evaluated -- one off-plane frame in eight should not move the result.

Findings below their operating threshold are returned under `not_detected` with their confidence,
so the reasoning agent can distinguish *actively excluded* from *never assessed*.


In [8]:
def predict_lung(frames):
    if 'lung' not in MODELS:
        return S.make_report('lung', [], status='not_supported')

    xs = torch.stack([_prep_cls(f) for f in frames]).to(device)
    with torch.no_grad():
        p_frame = torch.sigmoid(MODELS['lung'](xs)).cpu().numpy()
    p_clip = np.median(p_frame, axis=0)

    cal = CALIB.get('lung')
    thresholds = (cal or {}).get('thresholds', {n: 0.5 for n in LUNG_FINDINGS})
    unreliable = (cal or {}).get('unreliable_findings', [])

    detected, not_detected = [], []
    for i, name in enumerate(LUNG_FINDINGS):
        raw = float(p_clip[i])
        if cal and name in cal.get('platt', {}):
            co = cal['platt'][name]
            conf = apply_platt(raw, co['coef'], co['intercept'])
        else:
            conf = raw
        entry = S.make_finding(name.replace('_', ' '), conf)
        if name in unreliable:
            entry['unreliable'] = True
        (detected if raw > thresholds.get(name, 0.5) else not_detected).append(entry)

    # Both lists go through make_report so both are validated.
    #
    # A clip where nothing crosses threshold is a legitimate, informative result: the module
    # screened four findings and saw none of them. The schema represents that as an empty
    # `findings` beside a populated `not_detected`.
    #
    # It must NOT be represented by a placeholder entry in `findings`. Everything in that list
    # is marked detected=True by the Clinical State Builder, so a sentinel would be read as a
    # positive finding -- suppressing the "no positive finding" escalation trigger and counting
    # as weak evidence in the case-quality grade. The absence of findings would escalate less
    # readily than their presence, which is backwards.
    return S.make_report(
        'lung',
        sorted(detected, key=lambda f: -f['confidence']),
        not_detected=sorted(not_detected, key=lambda f: -f['confidence']),
        status='ok',
        quality={'no_finding_above_threshold': not detected},
        reliability={'confidence_calibrated': bool(cal),
                     'has_normal_class': False,
                     'modelled_findings': LUNG_FINDINGS,
                     'unreliable_findings': unreliable,
                     'scope': '187 clips / 165 cases; pneumothorax NOT modelled'},
        explanation={'type': 'grad_cam'},
        model='effnetb0_multilabel')


## 8. Routing

One entry point. Organs without a model return `not_supported` **in the same schema** rather than
raising, so the reasoning agent never branches on whether a key exists.


In [9]:
SUPPORTED = {'heart', 'gallbladder', 'lung'}


def ultrasound_agent(organ, **kwargs):
    """Route one study to its organ module.

    heart       -> ed_img, es_img   (two frames)
    gallbladder -> img              (one still)
    lung        -> frames           (list of frames from one clip)
    """
    organ = organ.lower()
    if organ not in S.ORGANS:
        raise ValueError(f'unknown organ {organ!r}; expected one of {sorted(S.ORGANS)}')
    if organ not in SUPPORTED:
        # vascular / FAST: defined interface, no model yet.
        return S.make_report(organ, [], status='not_supported',
                             reliability={'scope': 'module not implemented'})

    if organ == 'heart':
        return predict_heart(kwargs['ed_img'], kwargs['es_img'])
    if organ == 'gallbladder':
        return predict_gallbladder(kwargs['img'])
    return predict_lung(kwargs['frames'])


for organ in ['vascular', 'fast']:
    r = ultrasound_agent(organ)
    print(f'{organ:<12} -> status={r["status"]!r}, findings={r["findings"]}, '
          f'errors={S.validate_report(r) or "none"}')


vascular     -> status='not_supported', findings=[], errors=none
fast         -> status='not_supported', findings=[], errors=none


## 9. Exporting calibration from the module notebooks

If Section 2 reported a missing calibration file, add the matching cell to the end of that
module's notebook and re-run only that cell. Each is a few lines and requires no retraining --
the fitted objects are already in memory after the notebook's own evaluation sections.


In [10]:
EXPORT_SNIPPETS = {
'heart': """
# add to the end of cardiac_segmentation_training.ipynb (after Section 11)
xs = np.linspace(0, 1, 21)
json.dump({'isotonic_x': xs.tolist(),
           'isotonic_y': CALIBRATOR.predict(xs).tolist(),
           'ceiling': float(CALIBRATOR.predict([1.0])[0]),
           'ece': float(ece)},
          open(DRIVE_ROOT / 'Cardiac' / 'cardiac_calibration.json', 'w'), indent=2)
""",
'gallbladder': """
# add to the end of gallbladder_classification_training.ipynb (after Section 9)
torch.save(final_model.state_dict(),
           DRIVE_ROOT / 'Abdominal_Gallbladder' / 'gallbladder_effnetb0_best.pth')
json.dump({'temperature': float(T), 'ece': float(ece_cal)},
          open(DRIVE_ROOT / 'Abdominal_Gallbladder' / 'gallbladder_calibration.json', 'w'), indent=2)
""",
'lung': """
# add to the end of lung_cv_training.ipynb (after Section 11)
json.dump({'platt': {n: {'coef': float(CALIBRATORS[n].coef_[0][0]),
                         'intercept': float(CALIBRATORS[n].intercept_[0])}
                     for n in FINDING_COLS},
           'thresholds': {n.replace('finding_', ''): float(np.mean(chosen_t[n]))
                          for n in FINDING_COLS},
           'unreliable_findings': UNRELIABLE},
          open(DRIVE_ROOT / 'Pulmonary' / 'lung_calibration.json', 'w'), indent=2)
""",
}

needed = [o for o, a in AVAILABLE.items() if not a['calib']]
if not needed:
    print('All calibration artifacts present -- nothing to export.')
for organ in needed:
    print(f'{"=" * 70}\n{organ.upper()}{EXPORT_SNIPPETS[organ]}')


All calibration artifacts present -- nothing to export.


## 10. Encounter bundle

Triage and Ultrasound run in parallel and neither depends on the other, so each writes its own
section of a shared bundle. The Clinical Reasoning Agent reads the assembled file.

File-based rather than in-process because the organ models and an 8B reasoning LLM cannot occupy
one Colab session together -- the agents necessarily run at different times. Writes are atomic, so
a crash cannot leave a half-written bundle that still parses.


In [11]:
ENCOUNTER_ID = 'ENC-DEMO-001'

# Triage runs independently; its output is shown here as an example of what it contributes.
S.write_triage(ENCOUNTER_ID,
               S.make_triage('medium', 0.64, features={'HR': 104, 'SpO2': 93, 'RR': 24},
                             model='xgb_tier_v1'),
               ENCOUNTER_ROOT)
S.write_clinical(ENCOUNTER_ID,
                 {'age': 68, 'sex': 'M', 'symptoms': 'dyspnoea, pleuritic chest pain'},
                 ENCOUNTER_ROOT)

demo_frame = (np.random.RandomState(0).rand(300, 400) * 255).astype(np.uint8)

for organ, kw in [('lung', {'frames': [demo_frame] * 4}),
                  ('gallbladder', {'img': demo_frame}),
                  ('heart', {'ed_img': demo_frame, 'es_img': demo_frame}),
                  ('fast', {})]:
    rep = ultrasound_agent(organ, **kw)
    errs = S.validate_report(rep)
    if errs:
        print(f'{organ}: schema errors {errs}')
        continue
    S.write_ultrasound(ENCOUNTER_ID, rep, ENCOUNTER_ROOT)
    print(f'{organ:<12} status={rep["status"]:<15} findings={len(rep["findings"])}')

print(f'\nBundle: {ENCOUNTER_ROOT / "encounters" / (ENCOUNTER_ID + ".json")}')
print('NOTE: the demo uses random noise, so the findings below are meaningless. It exercises the')
print('routing, schema and storage path -- swap in a real clip to get a real report.')


lung         status=ok              findings=4
gallbladder  status=ok              findings=1
heart        status=failed          findings=0
fast         status=not_supported   findings=0

Bundle: /content/drive/MyDrive/POCUS-Project/encounters_store/encounters/ENC-DEMO-001.json
NOTE: the demo uses random noise, so the findings below are meaningless. It exercises the
routing, schema and storage path -- swap in a real clip to get a real report.


## 11. What the Clinical Reasoning Agent receives

Rendered deterministically to text rather than handed raw JSON: the same bundle always produces
byte-identical input, so the reasoning step is reproducible and auditable.

`CONFLICTS` is computed, not left for the LLM to notice. The architecture escalates when
confidence is low or when Triage and Ultrasound disagree, so both conditions are detected in code.


In [12]:
bundle = S.load_encounter(ENCOUNTER_ID, ENCOUNTER_ROOT)
print(S.render_for_prompt(bundle))


ENCOUNTER ENC-DEMO-001

PATIENT
  age: 68
  sex: M
  symptoms: dyspnoea, pleuritic chest pain

TRIAGE AGENT
  urgency: medium (confidence 0.64)

ULTRASOUND AGENT
  fast: not_supported
  gallbladder:
    - Acute cholecystitis (any severity) [Acute inflammation] (confidence 0.39)
    scope: teaching-atlas stills; no healthy class exists in the training data
    quality warnings: low_confidence
  heart: failed
  lung:
    - consolidation (confidence 0.98)
    - b lines (confidence 0.97)
    - pleural effusion (confidence 0.85)
    - pleural thickening (confidence 0.38)
    scope: 187 clips / 165 cases; pneumothorax NOT modelled

CONFLICTS
  - low confidence: gallbladder / Acute cholecystitis (any severity) at 0.39
  - low confidence: lung / pleural thickening at 0.38

LIMITS
  A 'normal' read means none of that model's trained classes fired. It does not
  rule out pathology the model was never trained to recognise.


## 12. Limitations

These travel with the agent's output, and the reasoning layer must respect them.

- **Small independent-case counts throughout.** Cardiac 100 validation patients, gallbladder 199
  cases, lung 165 cases. Every figure carries wide intervals; differences of a few points are not
  interpretable.
- **No organ has a healthy class.** Gallbladder was trained only on pathology chapters; lung
  models four findings and cannot see pneumothorax at all. **An empty finding list never means a
  normal study.**
- **Rare positives.** Pleural effusion has 24 positive clips and is flagged `unreliable`;
  cardiac's severe band has 11 cases.
- **Multi-label correlation.** Lung findings co-occur, so per-finding figures are not independent.
- **Curated public data, not bedside acquisitions.** Teaching-atlas stills and public clips are
  cleaner and better framed than real point-of-care imaging, and gallbladder frames carry
  burned-in annotation markers in 15-50% of cases.
- **Calibration is optimistic** where it was fitted and scored on the same out-of-fold
  predictions.
- **Grad-CAM is a plausibility check**, not verified localisation -- the gallbladder module in
  particular showed a pronounced central prior.
- **Further gains require more annotated data, not another architecture.** The single largest
  improvement in this project came from unfreezing a backbone (B-lines 0.512 -> 0.899), not from
  swapping models; several architecture and augmentation changes were tested and measurably did
  nothing.


In [13]:
print('=' * 72)
print('ULTRASOUND AGENT -- SUMMARY')
print('=' * 72)
rows = [
    ('heart', 'EfficientNet-B0 U-Net', 'LV/myo/LA Dice 0.93/0.83/0.89; EF MAE 7.1pp'),
    ('gallbladder', 'EfficientNet-B0', '5 classes, balanced acc 63.1% (chance 20%)'),
    ('lung', 'EfficientNet-B0 multi-label', '4 findings, macro AUROC 0.788'),
    ('vascular', '--', 'not implemented'),
    ('fast', '--', 'not implemented'),
]
for organ, model, result in rows:
    mark = 'ready' if organ in MODELS else ('stub' if organ not in SUPPORTED else 'NO WEIGHTS')
    print(f'  {organ:<12}{mark:<12}{model:<30}{result}')
print()
print(f'Schema version : {S.SCHEMA_VERSION}')
print(f'Encounters     : {ENCOUNTER_ROOT / "encounters"}')
print(f'Calibrated     : {sorted(CALIB) or "none -- confidences are RAW"}')
print()
print('This agent reports findings and confidence. Urgency, diagnosis and escalation are the')
print('Clinical Reasoning Agent\'s decisions, taken with clinical and biological data this')
print('agent never sees.')
print('=' * 72)


ULTRASOUND AGENT -- SUMMARY
  heart       ready       EfficientNet-B0 U-Net         LV/myo/LA Dice 0.93/0.83/0.89; EF MAE 7.1pp
  gallbladder ready       EfficientNet-B0               5 classes, balanced acc 63.1% (chance 20%)
  lung        ready       EfficientNet-B0 multi-label   4 findings, macro AUROC 0.788
  vascular    stub        --                            not implemented
  fast        stub        --                            not implemented

Schema version : 1.0
Encounters     : /content/drive/MyDrive/POCUS-Project/encounters_store/encounters
Calibrated     : ['gallbladder', 'heart', 'lung']

This agent reports findings and confidence. Urgency, diagnosis and escalation are the
Clinical Reasoning Agent's decisions, taken with clinical and biological data this
agent never sees.
